# Ranking-Based Anomaly Detection — End-to-end Demo

This notebook walks the full pipeline on **synthetic data** produced by `data/synthetic/generate.py`:

1. Generate a Wazuh-shaped synthetic dataset
2. Clean and split into T1 / T2 / T3
3. Inject MITRE-ATT&CK-style synthetic anomalies into T2
4. Fit feature transformers on T1, freeze them, apply to T2/T3
5. Train a k-NN scorer on T1
6. Score T3, inspect the top-K slice

Run from the repository root: `jupyter notebook notebooks/demo.ipynb`.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

ROOT = Path().resolve().parent
DATA = ROOT / "data" / "synthetic"
SPLITS = ROOT / "data" / "splits"
DATA.mkdir(parents=True, exist_ok=True)
SPLITS.mkdir(parents=True, exist_ok=True)

def run(cmd):
    """Run a shell command and stream output."""
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, cwd=str(ROOT), capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(r.stderr)

## 1. Generate synthetic Wazuh alerts

In [ ]:
run(f'{sys.executable} data/synthetic/generate.py --n 50000 --days 33 --start-date 2025-07-10 --seed 42 --out data/synthetic/wazuh.csv')

## 2. Clean and split

In [ ]:
# 01 merge is a no-op for a single file but we keep the full pipeline for parity
run(f'{sys.executable} src/pipeline/01_merge_csvs.py --input-dir data/synthetic --pattern "wazuh.csv" --out data/combined.csv')
run(f'{sys.executable} src/pipeline/03_clean.py --input data/combined.csv --output data/cleaned.csv --report-dir docs/cleaning')

# Split boundaries chosen so roughly 70/20/10 of synthetic data lands in T1/T2/T3
run(
    f'{sys.executable} src/pipeline/04_split_T1_T2_T3.py '
    f'--input data/cleaned.csv --out-dir data/splits '
    f'--history-cutoff 2025-07-31 --window-start 2025-08-01 --window-end 2025-08-07'
)

## 3. Inject synthetic anomalies into T2

In [ ]:
# The injection script expects the original Wazuh column names, so we point
# it at the source synthetic file with an explicit history cut-off.
run(
    f'{sys.executable} src/pipeline/05_inject_synthetic.py '
    f'--input data/synthetic/wazuh.csv '
    f'--history-cutoff 2025-07-31 --window-start 2025-08-01 --window-end 2025-08-07 '
    f'--out-window data/splits/T2_synth_window.csv '
    f'--out-for-fe data/splits/T2_synth_for_fe.csv '
    f'--rate-total 0.03'
)
print(pd.read_csv('../data/splits/T2_synth_window.csv')['synth_usecase'].value_counts())

## 4. Feature engineering — fit on T1, apply to T2/T3

In [ ]:
run(
    f'{sys.executable} src/pipeline/06_engineer_features.py fit '
    f'--input data/splits/T1.csv '
    f'--out data/splits/T1_feat.csv '
    f'--stats-out data/splits/fe_stats.json'
)
run(
    f'{sys.executable} src/pipeline/06_engineer_features.py apply '
    f'--input data/splits/T2.csv '
    f'--stats data/splits/fe_stats.json '
    f'--out data/splits/T2_feat.csv'
)
run(
    f'{sys.executable} src/pipeline/06_engineer_features.py apply '
    f'--input data/splits/T3.csv '
    f'--stats data/splits/fe_stats.json '
    f'--out data/splits/T3_feat.csv'
)

## 5. Train k-NN on T1 features and score T3

In [ ]:
FEATURES = [
    'hour_local', 'is_off_hours', 'day_of_week', 'is_weekend',
    'second_in_hour', 'seconds_in_hour_bucketed', 'hour_bucket',
    'rule_level', 'rule_level_z_host',
    'rule_time_since_last_host_log1p', 'recency_x_rule_level',
    'offhours_x_rule_level', 'offhours_x_rule_level_z', 'weekend_x_rule_level_z',
    'agent_rule_hour_hash01', 'agent_rule_hour_freq_log1p',
]

t1_feat = pd.read_csv('../data/splits/T1_feat.csv')
t3_feat = pd.read_csv('../data/splits/T3_feat.csv')

X_train = t1_feat[FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy()
X_test  = t3_feat[FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy()

knn = NearestNeighbors(n_neighbors=35, metric='euclidean', n_jobs=-1)
knn.fit(X_train)

dists, _ = knn.kneighbors(X_test)
scores = dists.mean(axis=1)
t3_feat['anomaly_score'] = scores

print(f'Trained on {len(X_train):,} rows, scored {len(X_test):,} rows.')
print(f'score min={scores.min():.4f}  mean={scores.mean():.4f}  max={scores.max():.4f}')

## 6. Inspect the top-K

In [ ]:
p = 0.01   # 1% daily operating budget
K = max(1, int(np.ceil(p * len(t3_feat))))
top = t3_feat.nlargest(K, 'anomaly_score').copy()
top['rank'] = np.arange(1, len(top) + 1)

cols = ['rank', 'event_id', 'timestamp', 'agent', 'rule_id', 'rule_level',
        'is_off_hours', 'rule_level_z_host', 'anomaly_score']
cols = [c for c in cols if c in top.columns]
top[cols].head(20)

## 7. Why did these alerts rank high? (simple explainability)

We compute per-feature occlusion: swap one feature's value for the training median and measure the score drop. Larger positive contribution = that feature is pushing this alert to the top.

In [ ]:
baseline = pd.DataFrame(X_train, columns=FEATURES).median()
X_top = top[FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy()
base_scores = knn.kneighbors(X_top)[0].mean(axis=1)

contrib = np.zeros((len(X_top), len(FEATURES)))
for j, col in enumerate(FEATURES):
    X_mod = X_top.copy()
    X_mod[:, j] = baseline[col]
    mod_scores = knn.kneighbors(X_mod)[0].mean(axis=1)
    contrib[:, j] = base_scores - mod_scores

# Show the top 3 drivers for the first 5 ranked alerts
for i in range(min(5, len(top))):
    order = np.argsort(-contrib[i])[:3]
    print(f'rank {i+1:>2}  top drivers: ', end='')
    print(', '.join(f'{FEATURES[j]}={contrib[i,j]:.3f}' for j in order))

## Next steps

- Swap the synthetic dataset for a larger one (`--n 500000`) and rerun to see how per-day precision stabilises.
- Tweak `--rate-total` and `--weights` in the injection step to see how the ranker handles different anomaly families.
- Start the FastAPI service (`uvicorn src.service.fastapi_app:app --port 8000`) and POST the scored T3 CSV to `/anomalies/top`.

For the thesis context (real data, real deployment, full results) see `README.md` and `docs/`.